<a href="https://colab.research.google.com/github/ricardoyx12/tareas-IA/blob/main/asignacion_2_regresion_ipynb_2_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Descripción del data set Financial data of 4400+ public companies

✅ ¿Qué es?

Se trata de un conjunto de datos que contiene información financiera de más de 4 400 empresas públicas.  

Los datos fueron raspados (“scraped”) de Yahoo Finance, lo que incluye los estados financieros — el balance general (“balance sheet”), el estado de resultados (“income statement”), y el flujo de caja (“cash flow statement”).  

Incluye datos tanto anuales como trimestrales para un período reciente (aproximadamente los últimos 4 años) para cada empresa.  


# ¿Qué contiene exactamente?

Algunas de las características clave de la base de datos:

- Cada empresa viene identificada con su ticker o símbolo bursátil (o equivalente) y posiblemente con su nombre, industria, etc.
- Para cada empresa, los datos incluyen:
    - Estado de resultados: ingresos, beneficios, gastos, etc.
    - Balance general: activos, pasivos, capital contable, etc.
    - Flujo de caja: flujos operativos, de inversión, de financiación, etc.
- Los datos están organizados para que puedas ver la evolución por trimestre y por año, lo que permite análisis de tendencias.
- Se pueden usar variables financieras como predictors (por ejemplo: activos, pasivos, ingresos) y variables objetivo como beneficio, rentabilidad, crecimiento, etc.

# Paso 1 — Reconocer el dataset

1. Objetivo: investigar el dataset entregado en la asignación 2, revisar el nombre de cada columna y documentar qué representa cada una (tipo, unidad, periodicidad, observaciones).

2. Pasos recomendados:
    - Cargar el archivo entregado (por ejemplo: df = pd.read_csv(...)) y listar columnas: df.columns
    - Para cada columna: buscar su significado (glosario, documentación de la fuente, o inspección de valores).
    - Anotar tipo de dato, unidad (USD, porcentaje, entero), periodicidad (trimestral/anual) y cualquier observación (por ejemplo: calculada, acumulada, neta/bruta).
    - Generar la lista final con nombre y descripción clara y breve.

3. Ejemplo de Plantilla para la lista (reemplazar con las columnas reales del dataset):

- ticker: Identificador bursátil de la empresa (string). Ejemplo: "AAPL".
- fiscal_date: Fecha del periodo financiero (YYYY-MM-DD). Indica el cierre del trimestre/año.
- revenue: Ingresos netos durante el periodo (num, USD). Periodicidad: trimestral/anual.
- gross_profit: Beneficio bruto (num, USD). Definición: ingresos menos costo de ventas.
- operating_income: Resultado operativo (num, USD). Incluye gastos operativos.
- net_income: Beneficio neto después de impuestos (num, USD).


4. Resultado esperado:
    - Un listado documentado con cada columna del dataset y su descripción (puede entregarse como tabla o como lista de pares nombre→descripción).

In [ ]:
import pandas as pd

# Reemplaza 'tu_archivo.csv' con el nombre real de tu dataset
df = pd.read_csv('tu_archivo.csv')

# Visualización de las columnas y tipos de datos para documentar
print("Columnas del dataset:")
print(df.columns.tolist())
print("\nInformación técnica:")
df.info()

# Ver los primeros registros para identificar unidades y periodicidad
df.head()

# Paso 2 — Seleccionar columnas relevantes

Después de haber cargado el dataset, elimina todas las columnas y quédate solo con: `stock`, `endDate` y `cash`.

- Verifica que los nombres y la capitalización de las columnas sean correctos antes de seleccionar.


In [ ]:
df['endDate'] = pd.to_datetime(df['endDate'])

# Paso 3 — Separar por empresas y elegir 3 para predecir su cantidad de dinero por fecha

- Objetivo: crear series temporales por empresa usando las columnas `stock`, `endDate` y `cash` y seleccionar 3 empresas para modelar y predecir `cash` por fecha.
- Pasos recomendados:
    - Verificar que las columnas `stock`, `endDate` y `cash` existen y están limpias (sin valores nulos o con imputación cuando sea necesario).
    - Ordenar el dataframe por `stock` y `endDate` (ascendente) para obtener la serie temporal de cada empresa.
    - Agrupar por `stock` y crear un subset por empresa.
    - Elegir 3 empresas con suficientes observaciones (p. ej. mayor número de fechas disponibles o relevancia del negocio).
    - Para cada empresa seleccionada, preparar los datos de entrenamiento/validación (features temporales, ventanas, lag, etc.) y definir la variable objetivo `cash` por `endDate`.

In [ ]:
 # Ordenar por stock y fecha para asegurar la secuencia temporal
df = df.sort_values(by=['stock', 'endDate'])

# Contar cuántas fechas tiene cada empresa y elegir las 3 con más datos
top_3_stocks = df['stock'].value_counts().nlargest(3).index.tolist()
print(f"Empresas seleccionadas: {top_3_stocks}")

# Crear un subset con solo estas 3 empresas
df_top3 = df[df['stock'].isin(top_3_stocks)].copy()

# Paso 4 — Grafique tiempo vs dinero de las 3 empresas en 3 gráficas diferentes

Objetivo: visualizar la serie temporal de `cash` frente a `endDate` para cada una de las 3 empresas seleccionadas, colocando cada empresa en una gráfica independiente.

Pasos recomendados:
- Asegurarse de tener el DataFrame con las columnas `stock`, `endDate` y `cash` y ordenado por `stock` y `endDate` ascendente.
- Convertir `endDate` a tipo fecha: `df['endDate'] = pd.to_datetime(df['endDate'])`.
- Seleccionar las 3 empresas elegidas: p. ej. `stocks = ['AAA','BBB','CCC']` y crear un subset por cada `stock`.
- Para cada empresa, graficar `endDate` en el eje x y `cash` en el eje y en una figura separada.
- Configurar títulos, etiquetas de ejes y formato de fechas (rotar etiquetas si es necesario). Añadir grid y leyenda si procede.
- Opcional: usar subplots (3 filas x 1 columna) para mostrar las 3 gráficas en la misma figura o generar 3 figuras individuales según preferencia.
- Guardar las figuras si es necesario: `plt.savefig('cash_stock_AAA.png', bbox_inches='tight')`.

Ejemplo de librerías a usar: matplotlib, seaborn o plotly para interactividad.

# Paso 5 — División 80/20 (entrenamiento / prueba)

Objetivo: separar los datos en 80% para entrenamiento y 20% para prueba respetando la estructura temporal por empresa (sin hacer shuffle).

Recomendaciones:
- Asegúrate de tener las columnas `stock`, `endDate` y `cash` y que el dataframe esté ordenado por `stock` y `endDate` (ascendente).
- Para series temporales por empresa, usa una división basada en tiempo: los primeros 80% de observaciones de cada `stock` → train; los últimos 20% → test.



In [ ]:
train_data = []
test_data = []

for stock in top_3_stocks:
    subset = df_top3[df_top3['stock'] == stock]
    split_idx = int(len(subset) * 0.8)
    train_data.append(subset.iloc[:split_idx])
    test_data.append(subset.iloc[split_idx:])

df_train = pd.concat(train_data)
df_test = pd.concat(test_data)

# Paso 6 — crea y entrena el modelo de regresión lineal para predecir `cash` por empresa

Objetivo: entrenar un modelo de regresión lineal por cada `stock` usando la serie temporal (respetando orden temporal) y evaluar en el 20% final.

Pasos recomendados:
- Preprocesamiento
    - Definir variables para el entrenamiento: y = `cash` (variable objetivo). `data` → X (conjunto de características a usar para predecir `cash`, p. ej. rezagos de `cash`, indicadores temporales, variables exógenas). Usar X e y en el entrenamiento: `model.fit(X_train, y_train)`.
    - Asegurar `endDate` como datetime y ordenar por `stock`, `endDate`.  - Asegurar `endDate` como datetime y ordenar por `stock`, `endDate`.



In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

models = {}

for stock in top_3_stocks:
    # Preparar datos de entrenamiento
    train_stock = df_train[df_train['stock'] == stock]
    X_train = train_stock['endDate'].map(pd.Timestamp.toordinal).values.reshape(-1, 1)
    y_train = train_stock['cash'].values

    # Entrenar modelo
    model = LinearRegression()
    model.fit(X_train, y_train)
    models[stock] = model

# Paso 7 — Verifica tu modelo de regresión lineal: grafica real vs predicho para las 3 empresas

- Objetivo: comparar visualmente los valores reales de `cash` del conjunto de prueba con los valores predichos por el modelo para cada una de las 3 empresas seleccionadas.
- Requisitos: tener `endDate` como datetime, el conjunto test por cada `stock`, y las predicciones (`y_pred`) para cada test.
- Pasos recomendados:
    - Para cada empresa (stock):
        - Extraer test: filas finales (20%) ordenadas por `endDate`.
        - Obtener predicciones usando el modelo entrenado: `y_pred = model.predict(X_test)`.
        - Crear una gráfica con `endDate` en el eje x y ambos: `cash` real (línea/points) y `cash` predicho (línea punteada) en el eje y.
        - Añadir título con el ticker, leyenda, etiquetas de ejes y grid. Formatear fechas y rotar etiquetas si hace falta.
    - Opcional: mostrar las 3 series en subplots (3 filas x 1 columna) para facilitar comparación.
    - Calcular y mostrar métricas de error por empresa (MAE, RMSE, R2) bajo cada gráfico o en una tabla resumen.
- Resultado esperado: tres gráficas (una por empresa) mostrando real vs predicho y una tabla o texto con las métricas de evaluación.

In [ ]:
for stock in top_3_stocks:
    test_stock = df_test[df_test[ 'stock'] == stock]
    X_test = test_stock['endDate'].map(pd.Timestamp.toordinal).values.reshape(-1, 1)
    y_test = test_stock['cash'].values

    # Predicción
    y_pred = models[stock].predict(X_test)

    plt.figure(figsize=(10, 4))
    plt.plot(test_stock['endDate'], y_test, label='Real', marker='o')
    plt.plot(test_stock['endDate'], y_pred, label='Predicho', linestyle='--', color='red')
    plt.title(f'Real vs Predicho: {stock}')
    plt.legend()
    plt.show()

# Paso 8 — Verificación de modelos (MSE, RMSE, R2)

Instrucciones breves:
- Asegúrese de tener para cada empresa: y_test (valores reales) y y_pred (predicciones).
- Calcular métricas con sklearn: mean_squared_error(y_test, y_pred), RMSE = sqrt(MSE), r2_score(y_test, y_pred).
- Presentar los resultados en una tabla por empresa y añadir una conclusión corta.

Empresa 1 (ticker: AAA)
| Métrica | Valor |
|---|---|
| MSE |  |
| RMSE |  |
| R2 |  |

Empresa 2 (ticker: BBB)
| Métrica | Valor |
|---|---|
| MSE |  |
| RMSE |  |
| R2 |  |

Empresa 3 (ticker: CCC)
| Métrica | Valor |
|---|---|
| MSE |  |
| RMSE |  |
| R2 |  |

Pequeña conclusión:
- Comparar RMSE/MSE absolutos para evaluar error en unidades de `cash`; RMSE más bajo = mejor precisión.
- R2 indica proporción de varianza explicada (cercano a 1 → buen ajuste; cercano a 0 o negativo → mal ajuste).
- Si alguno de los modelos muestra RMSE alto o R2 bajo, considerar: más features (rezagos, variables temporales), regularización, transformación de la serie o modelos no lineales.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

for stock in top_3_stocks:
    test_stock = df_test[df_test['stock'] == stock]
    X_test = test_stock['endDate'].map(pd.Timestamp.toordinal).values.reshape(-1, 1)
    y_test = test_stock['cash'].values
    y_pred = models[stock].predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print(f"--- {stock} ---")
    print(f"MSE: {mse:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R2: {r2:.4f}\n")

## Preguntas de Analisis

1. ¿Qué variables (features) usaste para predecir `cash` y por qué crees que son relevantes?

Se utilizó principalmente la variable temporal (endDate) convertida a formato numérico (ordinal) como característica principal. Esta variable es relevante porque en los datos financieros de empresas públicas, el flujo de caja suele presentar una tendencia o estacionalidad a lo largo del tiempo. El uso de fechas permite al modelo capturar si el efectivo de la empresa está creciendo o disminuyendo de manera constante en el periodo analizado




2. ¿Cómo realizaste la división temporal 80/20 por empresa y por qué es importante no barajar (shuffle) los datos en series temporales?

La división se realizó tomando el primer 80% de las observaciones cronológicas para el entrenamiento y el último 20% para la prueba de cada empresa. Es fundamental no barajar los datos porque las series temporales dependen estrictamente del orden secuencial. Si barajamos, el modelo podría "ver" datos del futuro para predecir el pasado, lo que se conoce como data leakage (filtración de datos), invalidando la capacidad del modelo para predecir eventos futuros reales


3. ¿Cuál es la diferencia entre MSE y RMSE y qué nos dice cada métrica sobre la precisión de las predicciones?

El MSE  mide el promedio de los errores al cuadrado, lo que penaliza de forma mucho más severa los errores grandes o "outliers". El RMSE es la raíz cuadrada del MSE y su principal ventaja es que devuelve la métrica a las mismas unidades que la variable original en este caso, dólares. Un RMSE bajo indica que, en promedio, las predicciones del modelo están cerca de los valores reales de efectivo




4. ¿Qué interpreta el valor de R² en este problema (predicción de `cash`) y qué limitaciones tiene su interpretación en series temporales?

El $R^2$ o coeficiente de determinación indica qué proporción de la varianza del cash es explicada por el modelo de regresión. Un valor cercano a 1 sugiere un buen ajuste a la tendencia. Sin embargo, en series temporales su limitación es que puede ser engañosamente alto si existe una tendencia muy marcada, incluso si el modelo no es capaz de capturar cambios bruscos o ciclos económicos volátiles




5. Si obtienes un RMSE alto o un R² bajo, ¿qué acciones concretas propondrías para mejorar el modelo (p. ej. features, rezagos, transformaciones, modelos alternativos)?

Para mejorar el desempeño del modelo, se podrían implementar las siguientes acciones:Añadir rezagos (lags): Usar el valor de cash del trimestre anterior como variable predictora.Variables exógenas: Incluir otras métricas financieras del dataset, como los ingresos (revenue) o el beneficio neto, que suelen estar correlacionados con el efectivo disponible.Modelos no lineales: Si la tendencia del efectivo no es una línea recta, se podrían probar modelos de regresión polinomial o modelos específicos de series temporales como ARIMA